# Imports

In [1]:
from fastai.vision.all import *
from pynvml import *
from pathlib import Path
import torch

# Variables

In [2]:
#portrait (height, width)
#image_size = (640,480) #512 #128 
#image_size = (480, 360)
image_size = (320, 240)
#image_size = (160, 120)
print(f"Image Size: {image_size}")

Image Size: (320, 240)


In [3]:
# This is so we have a uniform image size to process
resize = Resize(image_size, 
                 resamples=(Image.BILINEAR, Image.NEAREST),
                 method=ResizeMethod.Pad,
                 pad_mode=PadMode.Zeros) 

In [4]:
# Should we clean out the destination directories before processing?
clean = True
print(f"Clean: {clean}")

# Path to our training data
path = Path('./data')

path_sil = path/'silhouette'
path_sil_rawdata = path_sil/'rawdata'
path_sil_source = path_sil/'source' #lr
path_sil_mask = path_sil/'mask' #hr
path_sil_tattooless = path_sil/'tattooless'
path_sil_mask_predict = path_sil/'mask-predict'
path_sil_clipart = path_sil/'tattoo_clipart'

path_deinked = path/'deinked'
path_deinked_clean = path_deinked/"clean" #hr
path_deinked_tattoo = path_deinked/"tattoo" #lr
path_deinked_gen = path_deinked/"gen" # for placing our generated predictions

# Training batch size, tuning this based on the amount of memory
batch_size = 2
print(f"Batch Size: {batch_size}")

# TODO - what do we need to tune here?
wd, y_range, loss_gen = 1e-3, (-3., 3.), MSELossFlat()

Clean: True
Batch Size: 2


# Utility Functions

In [5]:
# Sanity checks for our GPU
def get_torch_device():  
    # Print out some metrics from our GPU just to make sure we can access it
    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)
    info = nvmlDeviceGetMemoryInfo(handle)
    print("Total memory:", info.total)
    print("Free memory:", info.free)
    print("Used memory:", info.used)

    #cuda configs
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")  
    print(f'CUDA is available: {torch.cuda.is_available()}')
    return device


# List all the available models
#import torchvision.models as tvmodels
#tvmodels.list_models()

In [6]:
# get_y function
# Note: this function will need to be recreated when the model is loaded for predictions
def _get_sil_y(x):
    return path_sil_mask/x.name

In [7]:
# This is for the training datablocks

# Helper method to create our datablock to load training data
def get_sil_dls(bs:int, size:int):
    dblock = DataBlock(blocks=(ImageBlock, ImageBlock),
                   get_items=get_image_files,
                   get_y = _get_sil_y,
                   splitter=RandomSplitter(),
                   item_tfms=Resize(
                       size, resamples=(Image.BILINEAR, Image.NEAREST),
                       method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
                   batch_tfms=[#*aug_transforms(max_zoom=2.),
                               Normalize.from_stats(*imagenet_stats)])
    dls = dblock.dataloaders(path_sil_source, bs=bs, path=path, num_workers=0, drop_last = True)
    dls.c = 3 # For 3 channel image
    #dls.drop_last=True
    return dls

In [8]:
# This is for the critic datablocks

# Create a composite between the source images and the generated
path_g = get_image_files(path_deinked_gen)
path_i = get_image_files(path_deinked_tattoo)
fnames = path_g + path_i

def get_crit_dls(fnames, bs:int, size:int):   
    "Generate two `Critic` DataLoaders"
    splits = RandomSplitter(0.1)(fnames)
    dsrc = Datasets(fnames, tfms=[[PILImage.create], [parent_label, Categorize]],
                 splits=splits)
    tfms = [ToTensor(), Resize(size)]
    gpu_tfms = [IntToFloatTensor(), Normalize.from_stats(*imagenet_stats)]
    dls = dsrc.dataloaders(bs=bs, after_item=tfms, after_batch=gpu_tfms, drop_last = True)
    #dls.drop_last=True
    return dls

In [9]:
# get_y function
# Note: this function will need to be recreated when the model is loaded for predictions
def _get_y(x):
    return path_deinked_clean/x.name

# Helper method to create our datablock to load training data
def get_dls(bs:int, size:int):
  dblock = DataBlock(blocks=(ImageBlock, ImageBlock),
                   get_items=get_image_files,
                   get_y = _get_y,
                   splitter=RandomSplitter(),
                   item_tfms=Resize(
                       size, resamples=(Image.BILINEAR, Image.NEAREST),
                       method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
                   batch_tfms=[#*aug_transforms(max_zoom=2.),
                               Normalize.from_stats(*imagenet_stats)])
  dls = dblock.dataloaders(path_deinked_tattoo, bs=bs, path=path, num_workers=0)
  dls.c = 3 # For 3 channel image
  return dls